In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Load all tables
orders    = pd.read_csv('../data/raw/olist_orders_dataset.csv')
items     = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
products  = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers   = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
reviews   = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
payments  = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
category  = pd.read_csv('../data/raw/product_category_name_translation.csv')

print("✓ All tables loaded")

✓ All tables loaded


In [2]:
from sqlalchemy import create_engine

# Test PostgreSQL connection
engine = create_engine('postgresql://paulamipaul@localhost:5432/paulamipaul')

try:
    with engine.connect() as conn:
        print("✓ PostgreSQL connected successfully")
        print("  Database: paulamipaul")
        print("  Port: 5432")
except Exception as e:
    print(f"✗ Connection failed: {e}")

✓ PostgreSQL connected successfully
  Database: paulamipaul
  Port: 5432


In [3]:
print("=== NULL AUDIT ===\n")
tables = {"orders":orders,"items":items,"customers":customers,
          "products":products,"sellers":sellers,
          "reviews":reviews,"payments":payments}

for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        print(f"[{name}]")
        for col, count in nulls.items():
            pct = round(count/len(df)*100, 1)
            print(f"  {col}: {count:,} nulls ({pct}%)")
    else:
        print(f"[{name}] no nulls")

=== NULL AUDIT ===

[orders]
  order_approved_at: 160 nulls (0.2%)
  order_delivered_carrier_date: 1,783 nulls (1.8%)
  order_delivered_customer_date: 2,965 nulls (3.0%)
[items] no nulls
[customers] no nulls
[products]
  product_category_name: 610 nulls (1.9%)
  product_name_lenght: 610 nulls (1.9%)
  product_description_lenght: 610 nulls (1.9%)
  product_photos_qty: 610 nulls (1.9%)
  product_weight_g: 2 nulls (0.0%)
  product_length_cm: 2 nulls (0.0%)
  product_height_cm: 2 nulls (0.0%)
  product_width_cm: 2 nulls (0.0%)
[sellers] no nulls
[reviews]
  review_comment_title: 87,656 nulls (88.3%)
  review_comment_message: 58,247 nulls (58.7%)
[payments] no nulls


In [4]:
date_cols = ['order_purchase_timestamp','order_approved_at',
             'order_delivered_carrier_date',
             'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_estimated_delivery_date']
).dt.days

orders['processing_time_days'] = (
    orders['order_approved_at'] -
    orders['order_purchase_timestamp']
).dt.days

print(f"✓ Orders cleaned")
print(f"  Late deliveries: {(orders['delivery_delay_days']>0).sum():,}")
print(f"  On time/early:   {(orders['delivery_delay_days']<=0).sum():,}")

✓ Orders cleaned
  Late deliveries: 6,535
  On time/early:   89,941


In [5]:
products = products.merge(category, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
for col in ['product_weight_g','product_length_cm','product_height_cm','product_width_cm']:
    products[col] = products[col].fillna(products[col].median())
print(f"✓ Products cleaned: {len(products):,} rows")

✓ Products cleaned: 32,951 rows


In [6]:
reviews['review_comment_title']   = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
reviews['review_creation_date']   = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
print(f"✓ Reviews cleaned: {len(reviews):,} rows")

✓ Reviews cleaned: 99,224 rows


In [7]:
master = (
    orders
    .merge(items, on='order_id', how='left')
    .merge(products[['product_id','product_category_name_english',
                      'product_weight_g']], on='product_id', how='left')
    .merge(customers[['customer_id','customer_city',
                       'customer_state']], on='customer_id', how='left')
    .merge(sellers[['seller_id','seller_city',
                    'seller_state']], on='seller_id', how='left')
    .merge(reviews[['order_id','review_score',
                    'review_comment_message']], on='order_id', how='left')
    .merge(payments.groupby('order_id')['payment_value']
           .sum().reset_index(), on='order_id', how='left')
)

print(f"✓ Master dataset: {len(master):,} rows, {master.shape[1]} columns")

✓ Master dataset: 114,092 rows, 25 columns


In [8]:
engine = create_engine('postgresql://paulamipaul@localhost:5432/paulamipaul')

# Load all tables into PostgreSQL
tables_to_load = {
    'orders_clean':    orders,
    'products_clean':  products,
    'customers':       customers,
    'sellers':         sellers,
    'reviews_clean':   reviews,
    'payments':        payments,
    'order_items':     items,
    'master_orders':   master
}

for table_name, df in tables_to_load.items():
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"✓ {table_name}: {len(df):,} rows loaded")

print("\n✓ All tables loaded into PostgreSQL")

✓ orders_clean: 99,441 rows loaded
✓ products_clean: 32,951 rows loaded
✓ customers: 99,441 rows loaded
✓ sellers: 3,095 rows loaded
✓ reviews_clean: 99,224 rows loaded
✓ payments: 103,886 rows loaded
✓ order_items: 112,650 rows loaded
✓ master_orders: 114,092 rows loaded

✓ All tables loaded into PostgreSQL


In [9]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as conn:
    result = pd.read_sql(text("""
        SELECT 
            product_category_name_english AS category,
            COUNT(*) AS total_orders,
            ROUND(AVG(review_score)::numeric, 2) AS avg_review,
            ROUND(SUM(payment_value)::numeric, 2) AS total_revenue
        FROM master_orders
        WHERE order_status = 'delivered'
        GROUP BY category
        ORDER BY total_revenue DESC
        LIMIT 10
    """), conn)

print(result)

                category  total_orders  avg_review  total_revenue
0         bed_bath_table         11107        3.92     1723932.14
1          health_beauty          9519        4.19     1625923.50
2  computers_accessories          7708        3.98     1563315.62
3        furniture_decor          8239        3.95     1408110.04
4          watches_gifts          5869        4.07     1388699.25
5         sports_leisure          8489        4.17     1357249.46
6             housewares          6819        4.11     1072820.85
7                   auto          4158        4.12      835782.91
8           garden_tools          4282        4.08      813055.77
9             cool_stuff          3727        4.19      746763.39
